# Hybrid QNN Training on CICIoT2023

This notebook trains a Hybrid Quantum Neural Network (QNN) on the CICIoT2023 dataset.
The architecture uses a classical projection layer to map high-dimensional input features (46+) down to 4 qubits for the quantum variational layer.

In [1]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import pennylane as qml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## 1. Configuration & Data Loading

In [11]:
# Configuration
DATASET_DIRECTORY = 'Datasets/unb-cic-iot-datase/wataiData/csv/CICIoT2023/'
FILES_TO_LOAD = 5  # Load only a subset for demonstration/speed
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 0.01
N_QUBITS = 4
N_LAYERS = 2
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

In [12]:
# Define Class Mapping (Binary: Attack vs Benign)
dict_2classes = {}
dict_2classes['DDoS-RSTFINFlood'] = 'Attack'
dict_2classes['DDoS-PSHACK_Flood'] = 'Attack'
dict_2classes['DDoS-SYN_Flood'] = 'Attack'
dict_2classes['DDoS-UDP_Flood'] = 'Attack'
dict_2classes['DDoS-TCP_Flood'] = 'Attack'
dict_2classes['DDoS-ICMP_Flood'] = 'Attack'
dict_2classes['DDoS-SynonymousIP_Flood'] = 'Attack'
dict_2classes['DDoS-ACK_Fragmentation'] = 'Attack'
dict_2classes['DDoS-UDP_Fragmentation'] = 'Attack'
dict_2classes['DDoS-ICMP_Fragmentation'] = 'Attack'
dict_2classes['DDoS-SlowLoris'] = 'Attack'
dict_2classes['DDoS-HTTP_Flood'] = 'Attack'

dict_2classes['DoS-UDP_Flood'] = 'Attack'
dict_2classes['DoS-SYN_Flood'] = 'Attack'
dict_2classes['DoS-TCP_Flood'] = 'Attack'
dict_2classes['DoS-HTTP_Flood'] = 'Attack'

dict_2classes['Mirai-greeth_flood'] = 'Attack'
dict_2classes['Mirai-greip_flood'] = 'Attack'
dict_2classes['Mirai-udpplain'] = 'Attack'

dict_2classes['Recon-PingSweep'] = 'Attack'
dict_2classes['Recon-OSScan'] = 'Attack'
dict_2classes['Recon-PortScan'] = 'Attack'
dict_2classes['VulnerabilityScan'] = 'Attack'
dict_2classes['Recon-HostDiscovery'] = 'Attack'

dict_2classes['DNS_Spoofing'] = 'Attack'
dict_2classes['MITM-ArpSpoofing'] = 'Attack'

dict_2classes['BenignTraffic'] = 'Benign'

dict_2classes['BrowserHijacking'] = 'Attack'
dict_2classes['Backdoor_Malware'] = 'Attack'
dict_2classes['XSS'] = 'Attack'
dict_2classes['Uploading_Attack'] = 'Attack'
dict_2classes['SqlInjection'] = 'Attack'
dict_2classes['CommandInjection'] = 'Attack'

dict_2classes['DictionaryBruteForce'] = 'Attack'

In [13]:
# Load Data
print("Loading dataset...")
csv_files = [f for f in os.listdir(DATASET_DIRECTORY) if f.endswith('.csv')]
csv_files.sort()
selected_files = csv_files[:FILES_TO_LOAD]

dfs = []
for file in tqdm(selected_files):
    path = os.path.join(DATASET_DIRECTORY, file)
    df = pd.read_csv(path)
    dfs.append(df)

full_df = pd.concat(dfs, ignore_index=True)
print(f"Total samples: {len(full_df)}")

Loading dataset...


100%|██████████| 5/5 [00:07<00:00,  1.41s/it]

Total samples: 1191264


In [14]:
# Preprocessing
X_columns = [
    'flow_duration', 'Header_Length', 'Protocol Type', 'Duration',
    'Rate', 'Srate', 'Drate', 'fin_flag_number', 'syn_flag_number',
    'rst_flag_number', 'psh_flag_number', 'ack_flag_number',
    'ece_flag_number', 'cwr_flag_number', 'ack_count',
    'syn_count', 'fin_count', 'urg_count', 'rst_count', 
    'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP',
    'UDP', 'DHCP', 'ARP', 'ICMP', 'IPv', 'LLC', 'Tot sum', 'Min',
    'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Magnitue',
    'Radius', 'Covariance', 'Variance', 'Weight', 
]
y_column = 'label'

# Map labels
full_df['binary_label'] = full_df[y_column].map(dict_2classes)
full_df['target'] = (full_df['binary_label'] == 'Attack').astype(int)

# Split Features and Target
X = full_df[X_columns].values
y = full_df['target'].values

# Scale Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=SEED)

# Convert to PyTorch Tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# Create DataLoaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print(f"Input Feature Dimension: {X_train.shape[1]}")

Training samples: 953011
Testing samples: 238253
Input Feature Dimension: 46


## 2. Hybrid QNN Architecture

In [15]:
# Quantum Device
dev = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev, interface="torch")
def qnn_layer(inputs, weights):
    # Encode classical data (Angle Embedding)
    # Inputs are expected to be of size N_QUBITS after projection
    for i in range(N_QUBITS):
        qml.RX(inputs[i], wires=i)

    # Variational layers
    qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    
    # Measurement
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

class HybridQNN(nn.Module):
    def __init__(self, input_dim, n_qubits, n_layers):
        super().__init__()
        self.n_qubits = n_qubits
        
        # Classical Projection Layer: Reduces input_dim -> n_qubits
        self.projection = nn.Linear(input_dim, n_qubits)
        
        # Quantum Weights
        self.q_weights = nn.Parameter(
            0.01 * torch.randn(n_layers, n_qubits, 3)
        )
        
        # Classical Output Head
        self.fc = nn.Linear(n_qubits, 1)

    def forward(self, x):
        # 1. Project High-Dim Features -> Qubits
        x_proj = torch.tanh(self.projection(x)) * np.pi # Scale to [-pi, pi] for RX gate
        
        # 2. Quantum Layer
        # We need to process the batch. qnn_layer takes a single sample.
        # Using a loop is slow but simple. For speed, we can use qml.qnode(..., interface='torch', diff_method='backprop') 
        # and rely on PennyLane's batching if supported, or just loop.
        # Here we use a loop for clarity and safety with the previous fix logic.
        
        q_out = []
        for sample in x_proj:
            q = qnn_layer(sample, self.q_weights)
            q = torch.stack(q) # Ensure it's a tensor
            q_out.append(q)
        
        q_out = torch.stack(q_out).float()
        
        # 3. Final Classification
        return self.fc(q_out)

In [16]:
# Initialize Model
input_dim = X_train.shape[1]
model = HybridQNN(input_dim, N_QUBITS, N_LAYERS)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.BCEWithLogitsLoss()

print(model)

HybridQNN(
  (projection): Linear(in_features=46, out_features=4, bias=True)
  (fc): Linear(in_features=4, out_features=1, bias=True)
)


## 3. Training Loop

In [ ]:
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            logits = model(X_batch)
            preds = torch.sigmoid(logits) > 0.5
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    return acc, prec, rec, f1

print("Starting training...")
for ep in range(EPOCHS):
    model.train()
    total_loss = 0
    for X_batch, y_batch in tqdm(train_loader, desc=f"Epoch {ep+1}/{EPOCHS}"):
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    acc, prec, rec, f1 = evaluate(model, test_loader)
    
    print(f"Epoch {ep+1}: Loss = {avg_loss:.4f}")
    print(f"  Test Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")

Starting training...


Epoch 1/5:  23%|██▎       | 6702/29782 [1:09:16<4:32:20,  1.41it/s]

In [ ]:
# Save the trained model
save_dir = 'trained_model'
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, 'hybrid_qnn_model.pth')
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")